# `safim_eval_1000` — Fixed Eval Set + Base vs Random-LoRA vs Distributed-LoRA
**Kaggle notebook — thin controller only. All logic lives in `scripts/` and `src/`.**

Two phases, both driven entirely by `configs/data/safim_eval_1000.yaml`:

- **Phase A (CPU)** — `scripts/generate_safim_eval_1000.py`: randomly selects
  ~750 files from `stack_v3_python_only_data_without_fim` — a small held-out
  pool that **continues the 10k pilot's `stack-v3-train` stream past
  `sample_filtered_data_10000`'s final checkpoint** (shard 9 / row 13500),
  so every file in it is strictly newer than anything the pilot scanned and
  hash-disjoint from it (see the `stack_v3_python_only_data_without_fim`
  notebook / `configs/data/stack_v3_eval_pool_no_fim.yaml`). Buckets them and
  samples exactly 1000 FIM tasks per the fixed quota (line=150, statement=150,
  expression=150, block=120, function-body=120, method-body=100, api-call=120,
  class-level=90), pushing the result to `experiment/safim_eval_1000/`.
  **Idempotent** — once that folder exists on HF, re-running this cell is a
  no-op. This set must stay fixed forever once used for a real comparison.
- **Phase B (GPU required)** — `scripts/run_safim_eval_1000.py`: evaluates
  Base, `experiment/random_model/seed42`, and `experiment/distributed_model/seed42`
  on those same 1000 fixed tasks (exact_match / edit_similarity, broken down
  by fim_type bucket). **Resumable** — each model's progress is checkpointed
  to a local JSONL and mirrored to `experiment/safim_eval_1000_results/` in
  the HF dataset repo every 50 records; re-running after a crash or session
  restart picks up mid-model instead of starting over.

Run Phase A first (works without a GPU). Switch to a GPU accelerator before
Phase B — it needs one for model inference.

> **Prerequisite:** the `stack_v3_python_only_data_without_fim` pool must
> already have ≥ `sampling.source_file_count` files pushed to HF — run the
> `stack_v3_python_only_data_without_fim` notebook first (it now resumes from
> the 10k pilot checkpoint, target 2,000 files).


In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────────────
import os
import shutil
import subprocess

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Cloning branch : {BRANCH or 'main'}")
print(f"Requested commit : {COMMIT or '(latest on branch)'}")

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

current_branch = subprocess.check_output(
    ["git", "-C", REPO_DIR, "branch", "--show-current"], text=True
).strip()
current_commit = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()

print("\n✓ Repository ready")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")


In [ ]:
# ── Cell 2: Install Phase-A dependencies (CPU-only — no torch needed yet) ───
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "datasets", "pyarrow", "huggingface_hub", "pyyaml"],
    check=True,
)


In [ ]:
# ── Cell 3: Authenticate to Hugging Face ────────────────────────
# HF_TOKEN is stored as a Kaggle Secret — NEVER hardcode tokens.
# Add it: Kaggle account → Settings → Secrets → Add New Secret
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")


In [ ]:
# ── Cell 4: GPU check (informational only — Phase A doesn't need one) ───────
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ GPU available — ready for Phase B once Phase A finishes.")
else:
    print("… No GPU on this session yet. Phase A below doesn't need one — "
          "switch accelerator + restart before running Phase B.")


## Phase A — build the fixed `safim_eval_1000` set (CPU)

In [ ]:
# ── Cell 5: Build the fixed eval set (idempotent — no-ops if it already exists) ──
import os

os.chdir(REPO_DIR)

subprocess.run(
    [sys.executable, "scripts/generate_safim_eval_1000.py",
     "--config", "configs/data/safim_eval_1000.yaml"],
    check=True,
)


In [ ]:
# ── Cell 6: Show the pushed eval set's metadata (bucket counts, shortfall) ──
import json
from huggingface_hub import hf_hub_download

meta_path = hf_hub_download(
    repo_id="Rudra-G-23/the-stack-v3-python-fim-data",
    filename="experiment/safim_eval_1000/metadata.json",
    repo_type="dataset",
    token=os.environ["HF_TOKEN"],
)
with open(meta_path, encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2))


## Phase B — evaluate Base / Random-LoRA / Distributed-LoRA (GPU required)

If Cell 4 showed no GPU, switch this notebook's accelerator on now, restart,
then re-run cells 1–4 before continuing.

In [ ]:
# ── Cell 7: Install Phase-B dependencies (GPU — model loading + inference) ──
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "unsloth", "peft", "trl", "transformers", "torch",
     "pandas", "seaborn", "matplotlib"],
    check=True,
)


In [ ]:
# ── Cell 8: Evaluate all three models on the fixed 1000 tasks ───────────────
# Resumable: if this crashes or the session restarts partway through, just
# re-run this cell — each model's already-scored tasks (local + HF-mirrored
# checkpoint) are skipped, not re-generated.
#
# For a fast first-time calibration (see notebook 0-cell's ETA note), try
# --max-samples 50 once, note the printed s/sample + ETA, then re-run without
# it for the full 1000.
subprocess.run(
    [sys.executable, "scripts/run_safim_eval_1000.py",
     "--config", "configs/data/safim_eval_1000.yaml"],
    check=True,
)


In [ ]:
# ── Cell 9: Display results ──────────────────────────────────────────────
import pandas as pd
from IPython.display import Image, display

output_dir = "results/safim_eval_1000"

print("Aggregate (Base vs Random-LoRA vs Distributed-LoRA):")
display(pd.read_csv(f"{output_dir}/metrics.csv"))

print("\nBy FIM bucket:")
display(pd.read_csv(f"{output_dir}/metrics_by_bucket.csv"))

for plot in ["comparison.png", "exact_match_by_bucket.png", "edit_similarity_by_bucket.png"]:
    display(Image(filename=f"{output_dir}/plots/{plot}"))
